# Day 159 — HuggingFace Transformers
## Month 9 | NLP + Deep Learning | Deepanshu Garg

| Field | Details |
|---|---|
| **Day** | 159 (Month 9, Week 2, Day 3) |
| **Topic** | HuggingFace Pipeline API → Zero-shot Sentiment → CLS Embeddings → LR Classifier |
| **Dataset** | ReviewPulse India (600 rows, seed=155) — review_text column |
| **Deliverable** | `Day159_HuggingFace_Transformers.ipynb` |
| **Max Score** | 80 pts + 10★ Bonus |
| **Environment** | Google Colab (Python 3.x) |

## Month 9 Scorecard

| Day | Topic | Score |
|---|---|---|
| Day 155 | Neural Networks & Keras | **90/90 + 10★ PERFECT** |
| Day 156 | CNNs | **80/80 + 10★ PERFECT** |
| Day 157 | RNNs & LSTMs | **80/80 + 10★ PERFECT** |
| Day 158 | NLP Text Processing | **—** |
| **Day 159** | **HuggingFace Transformers** | **— / 80** |

---

### Why HuggingFace Transformers?

TF-IDF and BoW count words. Transformers **understand context**.
'Not bad' → TF-IDF sees 'bad' (negative). Transformer sees the phrase as positive.
HuggingFace gives you pre-trained BERT/DistilBERT models in 3 lines of code.

**Today's pipeline:**
```
Raw text → HF Tokenizer → DistilBERT → [CLS] token → label / embedding
```

**Two modes you'll use:**
1. `pipeline('sentiment-analysis')` — zero-shot, no training needed
2. `AutoModel` → extract CLS embeddings → train LR on top (transfer learning)


---
## Section 1 — Raw Data Lock
> **DO NOT MODIFY THIS SECTION.**

In [1]:
# SECTION 1 — Raw Data Generation (seed=155, DO NOT MODIFY)
import numpy as np
import pandas as pd
import random

np.random.seed(155)
random.seed(155)
N = 600

platforms     = ['Upwork','Fiverr','Toptal','Freelancer','Contra']
skills        = ['Python','SQL','Tableau','Power BI','Excel','ML','NLP','Streamlit']
client_types  = ['Startup','SME','Enterprise','NGO']
project_types = ['Dashboard','Analysis','Automation','Model','Report']

platform             = np.random.choice(platforms,    N)
skill                = np.random.choice(skills,       N)
client_type          = np.random.choice(client_types, N)
project_type         = np.random.choice(project_types,N)
experience_years     = np.round(np.random.uniform(0.5, 8, N), 1)
hourly_rate          = np.round(np.random.uniform(10, 60, N), 2)
project_duration_days= np.random.randint(3, 120, N)
client_rating        = np.round(np.random.uniform(3.0, 5.0, N), 1)
num_revisions        = np.random.randint(0, 6, N)
communication_score  = np.random.randint(1, 6, N)

pos_templates = [
    'Excellent work, delivered on time and exceeded expectations.',
    'Great communication and professional output throughout the project.',
    'Highly skilled analyst, would definitely hire again.',
    'Outstanding quality, very responsive and easy to work with.',
    'Delivered clean insightful dashboards, very satisfied.',
    'Top notch work, precise and well documented.',
    'Amazing experience, fast turnaround and great results.',
]
neg_templates = [
    'Missed the deadline and quality was below expectations.',
    'Poor communication, had to follow up multiple times.',
    'Work was incomplete and not what was discussed initially.',
    'Disappointing results, did not meet the project requirements.',
    'Average quality with several errors in the analysis.',
    'Slow progress and unclear documentation delivered.',
]

hired_again_raw = (
    0.3*(client_rating-3.0)/2.0 +
    0.2*(communication_score-1)/4.0 +
    0.2*(1-num_revisions/5.0) +
    0.15*(experience_years/8.0) +
    0.15*np.random.uniform(0,1,N)
)
hired_again = (hired_again_raw > 0.5).astype(int)

review_text = []
for i in range(N):
    if hired_again[i]==1: review_text.append(random.choice(pos_templates))
    else:                 review_text.append(random.choice(neg_templates))

df_raw = pd.DataFrame({
    'platform':platform, 'skill':skill, 'client_type':client_type,
    'project_type':project_type, 'experience_years':experience_years,
    'hourly_rate':hourly_rate, 'project_duration_days':project_duration_days,
    'client_rating':client_rating, 'num_revisions':num_revisions,
    'communication_score':communication_score,
    'review_text':review_text, 'hired_again':hired_again
})

print('Shape          :', df_raw.shape)
print('Columns        :', list(df_raw.columns))
print('hired_again    :', df_raw.hired_again.value_counts().to_dict())
print('Unique reviews :', df_raw.review_text.nunique())
df_raw.head(3)

Shape          : (600, 12)
Columns        : ['platform', 'skill', 'client_type', 'project_type', 'experience_years', 'hourly_rate', 'project_duration_days', 'client_rating', 'num_revisions', 'communication_score', 'review_text', 'hired_again']
hired_again    : {0: 314, 1: 286}
Unique reviews : 13


,platform,skill,client_type,project_type,experience_years,hourly_rate,project_duration_days,client_rating,num_revisions,communication_score,review_text,hired_again
0,Contra,Streamlit,Startup,Dashboard,3.5,18.05,43,4.5,4,5,"Delivered clean insightful dashboards, very sa...",1
1,Fiverr,NLP,Startup,Automation,1.2,15.81,75,4.7,4,4,"Delivered clean insightful dashboards, very sa...",1
2,Upwork,Power BI,SME,Automation,5.3,29.71,113,3.3,3,1,Average quality with several errors in the ana...,0


---
## Section 2 — Concept Notes

### The NLP Progression

| Method | Knows Word Order? | Context-aware? | Training needed? |
|---|---|---|---|
| BoW / TF-IDF | ❌ | ❌ | Yes (on your data) |
| RNN / LSTM | ✅ | Partial | Yes |
| **Transformer (BERT)** | ✅ | ✅ Full bidirectional | **Pre-trained** |

### HuggingFace Pipeline API
```python
from transformers import pipeline
clf = pipeline('sentiment-analysis')   # downloads distilbert by default
clf('Great work, very satisfied!')     # → [{'label':'POSITIVE','score':0.9998}]
```
- `label` : POSITIVE / NEGATIVE
- `score` : model confidence (softmax probability)

### CLS Token Embeddings
Every BERT/DistilBERT output has a special `[CLS]` token at position 0.
Its 768-dim hidden state = **sentence-level representation**.
You can use it as features for any downstream classifier.

```
text → tokenize → DistilBERT → last_hidden_state[:,0,:] → (768,) vector
```

### Transfer Learning Intuition
The model was pre-trained on 11GB of text. It already knows:
- Grammar, syntax, semantics
- 'Not bad' ≠ 'bad'
- Domain sentiment nuances

We **freeze** those weights and just train a tiny LR on top of the CLS embeddings.
This is called **feature-based transfer learning**.

### Speed Tip for Colab
- Use `batch_size=32` in the pipeline for faster inference
- Use `torch.no_grad()` during embedding extraction
- Run on a 200-row subset if GPU is unavailable (CPU inference is slow)


---
## Section 3 — Setup & Imports

In [2]:
# SETUP — Run this cell first (Colab will install if needed)
# !pip install transformers torch -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

# Work on a copy — never touch df_raw
df = df_raw.copy()

TFIDF_ACC = None  # will hold Day 158 baseline for comparison
print('Setup complete. torch version:', torch.__version__)
print('Rows:', len(df), '| Columns:', len(df.columns))

Setup complete. torch version: 2.11.0+cpu
Rows: 600 | Columns: 12


---
## Section 4 — Practice Tasks
> Work independently. Do not look at Section 5 until done.

| Task | Topic | Points |
|---|---|---|
| T1 | Load HF sentiment pipeline, inspect outputs on 3 reviews | 10 |
| T2 | Run pipeline on 10-sample batch, build result DataFrame | 15 |
| T3 | Full test-set zero-shot accuracy + NRA insight | 15 |
| T4 | Extract CLS embeddings (200-row subset) | 20 |
| T5 | Train LR on CLS embeddings, compare vs TF-IDF baseline + NRA | 20 |
| **Bonus★** | Write architecture comparison diagram (markdown table) | 10★ |
| **Total** | | **80 + 10★** |

### Task 1 — Load HF Pipeline (10 pts)
1. Load `pipeline('sentiment-analysis')` into variable `clf`.
   - Model: `'distilbert-base-uncased-finetuned-sst-2-english'` (specify explicitly)
2. Run it on these 3 texts:
   - `'Great communication and professional output throughout the project.'`
   - `'Missed the deadline and quality was below expectations.'`
   - `'Average quality with several errors in the analysis.'`
3. Print: text | label | score (rounded to 4dp) for each.
4. Confirm: are the labels logically consistent with `hired_again`?

> **Expected:** All 3 correctly classified. Positive text → POSITIVE label, negative texts → NEGATIVE.

In [3]:
# ── T1: Load HuggingFace sentiment pipeline ─────────────────────────────
# Goal: Instantiate the zero-shot sentiment pipeline and test it on three sample reviews.
# Method: Use pipeline('sentiment-analysis') with the specified DistilBERT model.
#         Run inference on three texts and print label + confidence.

# Step 1: Load pipeline with explicit model name
clf = pipeline('sentiment-analysis',
               model='distilbert-base-uncased-finetuned-sst-2-english')

# Step 2: Define 3 test texts
t1_texts = [
    'Great communication and professional output throughout the project.',
    'Missed the deadline and quality was below expectations.',
    'Average quality with several errors in the analysis.',
]

# Step 3: Run inference + print results
t1_results = clf(t1_texts)
for txt, res in zip(t1_texts, t1_results):
    print(f"Text   : {txt[:60]}...")
    print(f"Label  : {res['label']}  |  Score: {res['score']:.4f}")
    print()

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Text   : Great communication and professional output throughout the p...
Label  : POSITIVE  |  Score: 0.9999

Text   : Missed the deadline and quality was below expectations....
Label  : NEGATIVE  |  Score: 0.9992

Text   : Average quality with several errors in the analysis....
Label  : NEGATIVE  |  Score: 0.9977



### Task 2 — Batch Inference on 10 Samples (15 pts)
1. Sample 10 rows from `df` with `random_state=42`.
2. Run `clf` on the `review_text` column (pass as list).
3. Build a DataFrame `t2_df` with columns:
   `review_text | hired_again | hf_label_str | hf_score | hf_label_int | correct`
   - `hf_label_str`: raw output ('POSITIVE' / 'NEGATIVE')
   - `hf_score`: confidence (float, 4dp)
   - `hf_label_int`: 1 if POSITIVE else 0
   - `correct`: 1 if hf_label_int == hired_again else 0
4. Print the DataFrame and the sample accuracy.

> **Expected:** 10/10 correct. Sample accuracy = 1.0000.

In [4]:
# ── T2: Batch inference on a 10‑row sample ──────────────────────────────
# Goal: Test the pipeline on a small sample, build a result DataFrame with
#       labels, scores, and correctness flags.
# Method: Sample 10 rows with fixed seed, run clf on the list of texts,
#         map labels to integers, and compute accuracy.

sample10 = df.sample(10, random_state=42).reset_index(drop=True)

# Run pipeline on review_text list
t2_results = clf(sample10['review_text'].tolist())

# Build result DataFrame
t2_df = sample10[['review_text','hired_again']].copy()
t2_df['hf_label_str'] = [r['label'] for r in t2_results]
t2_df['hf_score']     = [round(r['score'],4) for r in t2_results]
t2_df['hf_label_int'] = t2_df['hf_label_str'].map({'POSITIVE':1,'NEGATIVE':0})
t2_df['correct']      = (t2_df['hf_label_int'] == t2_df['hired_again']).astype(int)

print(t2_df.to_string())
print(f"\nSample Accuracy: {t2_df['correct'].mean():.4f}")

                                                           review_text  hired_again hf_label_str  hf_score  hf_label_int  correct
0            Work was incomplete and not what was discussed initially.            0     NEGATIVE    0.9998             0        1
1          Outstanding quality, very responsive and easy to work with.            1     POSITIVE    0.9999             1        1
2              Missed the deadline and quality was below expectations.            0     NEGATIVE    0.9992             0        1
3  Great communication and professional output throughout the project.            1     POSITIVE    0.9999             1        1
4                   Slow progress and unclear documentation delivered.            0     NEGATIVE    0.9998             0        1
5                 Average quality with several errors in the analysis.            0     NEGATIVE    0.9977             0        1
6                 Average quality with several errors in the analysis.            0     NE

### Task 3 — Test-Set Zero-Shot Accuracy + NRA (15 pts)
1. Create the standard 80/20 split with `random_state=155`, `stratify=y`:
   - `X = df['review_text']`, `y = df['hired_again']`
2. Run `clf` on `X_test` (as list). Map POSITIVE→1, NEGATIVE→0.
3. Compute and print:
   - Zero-shot accuracy on test set
   - `classification_report` (precision, recall, F1)
4. Write an NRA insight:
   - Number = exact zero-shot accuracy
   - Reason = why transformer gets this right
   - Action = specific business decision this enables

> **Expected:** Zero-shot accuracy = 1.0000 (text perfectly maps to labels by construction).

In [8]:
# ── T3: Zero‑shot accuracy on the full test set ─────────────────────────
# Goal: Evaluate the pipeline's ability to classify reviews without any training.
# Method: Split data 80/20, pass test texts to clf, map predictions, compute
#         accuracy and classification report. Provide an NRA insight.

X = df['review_text']
y = df['hired_again']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=155, stratify=y
)

# Run zero-shot on test set
t3_results = clf(X_test.tolist(), batch_size=32)
hf_preds = [1 if r['label']=='POSITIVE' else 0 for r in t3_results]

zs_acc = accuracy_score(y_test, hf_preds)
print(f'Zero-Shot Test Accuracy : {zs_acc:.4f}')
print(f'Test set size           : {len(y_test)}')
print()
print(classification_report(y_test, hf_preds, target_names=['Not Hired','Hired Again']))

# NRA Insight
print('=== NRA INSIGHT ===')
print(f'Number : Zero-shot accuracy = {zs_acc:.4f}')
print('Reason : DistilBERT was pre‑trained on 11GB of text, learning bidirectional contextual representations that encode sentiment polarity without any labelled data from our dataset. The pre‑tuned sentiment head directly maps the [CLS] token’s representation to POSITIVE/NEGATIVE.')
print('Action : Deploy this zero‑shot pipeline as a rapid screening tool for incoming reviews; if test accuracy drops below 0.95 on real data, schedule a CLS‑embedding fine‑tuning sprint.')


Zero-Shot Test Accuracy : 1.0000
Test set size           : 120

              precision    recall  f1-score   support

   Not Hired       1.00      1.00      1.00        63
 Hired Again       1.00      1.00      1.00        57

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120

=== NRA INSIGHT ===
Number : Zero-shot accuracy = 1.0000
Reason : DistilBERT was pre‑trained on 11GB of text, learning bidirectional contextual representations that encode sentiment polarity without any labelled data from our dataset. The pre‑tuned sentiment head directly maps the [CLS] token’s representation to POSITIVE/NEGATIVE.
Action : Deploy this zero‑shot pipeline as a rapid screening tool for incoming reviews; if test accuracy drops below 0.95 on real data, schedule a CLS‑embedding fine‑tuning sprint.


### Task 4 — Extract CLS Embeddings (20 pts)
Goal: Turn each review into a 768-dim vector using DistilBERT's `[CLS]` token.

1. Load tokenizer and model:
   - `AutoTokenizer.from_pretrained('distilbert-base-uncased')`
   - `AutoModel.from_pretrained('distilbert-base-uncased')`
2. Use `df.sample(200, random_state=155)` as `df_sub`.
3. Write function `get_cls_embeddings(texts, tokenizer, model, batch_size=32)`:
   - Tokenize with `padding=True, truncation=True, max_length=64, return_tensors='pt'`
   - Forward pass inside `torch.no_grad()`
   - Extract `last_hidden_state[:, 0, :]` (CLS token)
   - Return `numpy` array
4. Call it on `df_sub['review_text']`. Store result in `cls_embeddings`.
5. Print: shape of `cls_embeddings`, min value, max value (all rounded to 4dp).

> **Expected shape:** (200, 768). Min/max will vary by batch but typically in range [-2, 2].

In [6]:
# ── T4: Extract CLS token embeddings from DistilBERT ────────────────────
# Goal: Convert each review into a 768‑dimensional vector using the [CLS]
#       token of DistilBERT.
# Method: Load tokenizer and model, define a batching function that tokenizes,
#         runs a forward pass under torch.no_grad(), and returns the CLS vector.

MODEL_NAME = 'distilbert-base-uncased'

# Step 1: Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

# Step 2: Subset
df_sub = df.sample(200, random_state=155).reset_index(drop=True)

# Step 3: Embedding function
def get_cls_embeddings(texts, tokenizer, model, batch_size=32):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Tokenize
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors='pt'
        )
        # Forward pass — no gradients needed
        with torch.no_grad():
            outputs = model(**encoded)
        # Extract CLS token (position 0)
        cls_vec = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_vec)
    return np.vstack(all_embeddings)

# Step 4: Get embeddings
cls_embeddings = get_cls_embeddings(df_sub['review_text'].tolist(), tokenizer, model)

# Step 5: Print shape and stats
print(f'cls_embeddings shape : {cls_embeddings.shape}')
print(f'Min value            : {cls_embeddings.min():.4f}')
print(f'Max value            : {cls_embeddings.max():.4f}')
print(f'Mean value           : {cls_embeddings.mean():.4f}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cls_embeddings shape : (200, 768)
Min value            : -7.8580
Max value            : 3.5965
Mean value           : -0.0088


### Task 5 — LR on CLS Embeddings vs TF-IDF Baseline (20 pts)
1. Use `df_sub` (200 rows) and `cls_embeddings` from T4.
2. Get labels: `y_sub = df_sub['hired_again'].values`
3. Split 80/20 (`random_state=155`, `stratify=y_sub`).
4. Train `LogisticRegression(max_iter=500, random_state=155)` on CLS features.
5. Also run TF-IDF baseline on the same 200-row subset (same split).
6. Print a comparison table:

```
Method              | Train Acc | Test Acc
TF-IDF + LR         |  X.XXXX   |  X.XXXX
CLS Embedding + LR  |  X.XXXX   |  X.XXXX
```

7. Write NRA insight comparing both approaches for a real freelance client context.

> **Expected:** Both will achieve near-perfect accuracy on this synthetic dataset.

In [9]:
# ── T5: Compare CLS‑embedding + LR vs TF‑IDF + LR ──────────────────────
# Goal: Train a logistic regression on the extracted CLS vectors and contrast
#       its performance against a traditional TF‑IDF baseline on the same subset.
# Method: Split the 200‑row subset, train both models, print a comparison table,
#         and provide an NRA insight.

y_sub = df_sub['hired_again'].values

# Split for CLS
X_tr_cls, X_te_cls, y_tr_cls, y_te_cls = train_test_split(
    cls_embeddings, y_sub, test_size=0.2, random_state=155, stratify=y_sub
)

# Train LR on CLS embeddings
lr_cls = LogisticRegression(max_iter=500, random_state=155)
lr_cls.fit(X_tr_cls, y_tr_cls)
cls_train_acc = accuracy_score(y_tr_cls, lr_cls.predict(X_tr_cls))
cls_test_acc  = accuracy_score(y_te_cls, lr_cls.predict(X_te_cls))

# TF-IDF baseline on same 200-row subset
X_tr_txt, X_te_txt, y_tr_txt, y_te_txt = train_test_split(
    df_sub['review_text'], y_sub, test_size=0.2, random_state=155, stratify=y_sub
)
tfidf_sub = TfidfVectorizer(max_features=500, stop_words='english')
X_tr_tfidf = tfidf_sub.fit_transform(X_tr_txt)
X_te_tfidf = tfidf_sub.transform(X_te_txt)
lr_tfidf = LogisticRegression(max_iter=500, random_state=155)
lr_tfidf.fit(X_tr_tfidf, y_tr_txt)
tfidf_train_acc = accuracy_score(y_tr_txt, lr_tfidf.predict(X_tr_tfidf))
tfidf_test_acc  = accuracy_score(y_te_txt, lr_tfidf.predict(X_te_tfidf))

# Comparison table
print(f"{'Method':<25} | {'Train Acc':^10} | {'Test Acc':^10}")
print('-'*52)
print(f"{'TF-IDF + LR':<25} | {tfidf_train_acc:^10.4f} | {tfidf_test_acc:^10.4f}")
print(f"{'CLS Embedding + LR':<25} | {cls_train_acc:^10.4f} | {cls_test_acc:^10.4f}")

# Store for NRA
TFIDF_ACC = tfidf_test_acc

# NRA Insight
print('\n=== NRA INSIGHT ===')
print(f'Number : CLS LR test acc = {cls_test_acc:.4f} vs TF-IDF = {tfidf_test_acc:.4f}')
print('Reason : CLS embeddings outperform TF-IDF when reviews contain negation or sarcasm, because bidirectional attention encodes full‑sentence meaning rather than isolated word frequencies. TF‑IDF only counts words, so it misses “not bad” as positive; CLS captures the contextual flip.')
print('Action : Use CLS embeddings for the production client‑feedback classifier; if the TF‑IDF accuracy falls below 0.90 on real data, switch to DistilBERT features without retraining the transformer.')


Method                    | Train Acc  |  Test Acc 
----------------------------------------------------
TF-IDF + LR               |   1.0000   |   1.0000  
CLS Embedding + LR        |   1.0000   |   1.0000  

=== NRA INSIGHT ===
Number : CLS LR test acc = 1.0000 vs TF-IDF = 1.0000
Reason : CLS embeddings outperform TF-IDF when reviews contain negation or sarcasm, because bidirectional attention encodes full‑sentence meaning rather than isolated word frequencies. TF‑IDF only counts words, so it misses “not bad” as positive; CLS captures the contextual flip.
Action : Use CLS embeddings for the production client‑feedback classifier; if the TF‑IDF accuracy falls below 0.90 on real data, switch to DistilBERT features without retraining the transformer.


### ★ Bonus — Architecture Comparison Table (10★)
In a markdown cell, write a comparison table covering ALL four methods you've used in Month 9 for text/classification:

Columns: `Method | Input Format | Knows Context? | Pre-trained? | Training Speed | When to Use`

Rows: `TF-IDF + LR | LSTM | Zero-shot HF Pipeline | CLS Embedding + LR`

Then write 2 sentences: **When would you recommend a client use a Transformer over TF-IDF?**
Must be client-language (no jargon), must name a measurable condition.

#### YOUR BONUS ANSWER

| Method | Input Format | Knows Context? | Pre-trained? | Training Speed | When to Use |
|---|---|---|---|---|---|
| TF-IDF + LR | Word counts | ❌ | ❌ | Fast | Quick baseline, short documents, when interpretability matters |
| LSTM | Sequences | Partial | ❌ | Slow | When order and short‑range dependencies matter, but you have >10k labelled examples |
| Zero-shot HF Pipeline | Raw text | ✅ | ✅ | Instant | No training data, rapid prototyping, or when you need a single model for many tasks |
| CLS Embedding + LR | 768‑dim vector | ✅ | ✅ (frozen) | Medium | Production classification with moderate labelled data (hundreds to thousands of examples) |

**Client recommendation:** Use a transformer (CLS + LR) if your dataset has more than 500 labelled reviews and you care about nuances like negations; otherwise, start with TF‑IDF for a fast, interpretable MVP and only upgrade when accuracy drops below 85% on your validation set.

---
## Section 5 — Scoring Rubric

| Task | What's Checked | Points |
|---|---|---|
| **T1** | clf loaded with correct model string; 3 texts inferred; label+score printed; logical consistency confirmed | 10 |
| **T2** | sample(10,random_state=42); all 5 columns present; correct mapping POSITIVE→1/NEGATIVE→0; accuracy printed = 1.0000 | 15 |
| **T3** | Correct split (stratify=y, seed=155); zs_acc = 1.0000; classification_report shown; NRA has Number+Reason+Action (named, no hedging) | 15 |
| **T4** | Both tokenizer + model loaded from 'distilbert-base-uncased'; batch loop correct; torch.no_grad() used; shape = (200,768); min/max/mean printed | 20 |
| **T5** | Same 200-row split for both methods (seed=155, stratify); LR trained on CLS correctly; comparison table printed; NRA names specific F1 threshold | 20 |
| **★ Bonus** | Table complete (4 rows, 6 cols); all entries correct; client recommendation has measurable condition and no jargon | 10★ |
| **Total** | | **80 + 10★** |

### NRA Grading Standard
- ✅ `Number`: exact value from printed output (not 'high', not 'about 1.0')
- ✅ `Reason`: mechanism (why it works), not description of what happened
- ✅ `Action`: names a specific model/tool/threshold, commits to a decision — no 'consider', no 'might'

### Interview Answer
*'Explain HuggingFace transformers to a client who asked why their keyword-based system keeps misclassifying reviews.'*

**Answer:** Your current system counts words — if it sees 'bad', it flags negative. But it misses things like 'not bad at all' or 'better than expected'. HuggingFace transformers read the full sentence the same way a human does — left, right, and across clauses simultaneously. I pre-trained a model on 11 billion words so it already understands English sentiment. We plug your reviews in and get predictions immediately, without needing you to label thousands of examples first.